In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from crewai import LLM

llm=LLM(
    model="gemini/gemini-2.5-flash",
    temperature=0.1
)

In [ ]:
original_email= """ 
looping in Priya. TAS and PRX updates are in the deck. ETA for SDS integration is Friday.
Let's sync up tomorrow if SYNCBOT allows. ping me if any blockers.
"""
from crewai.tools import BaseTool

class ReplaceJargonsTool(BaseTool):
    name: str="Replace Jargons"
    description:str="Replace jargons withmore specific terms"
    
    def _run(self,email:str)->str:
        replacements={
            "PRX":"Project Phoenix (internal AI revamp project)",
            "TAS": "technical architecture stack",
            "DBX": "client database cluster",
            "SDS": "Smaert Data Syncer",
            "SYNCBOT":"internal standup assistant bot",
            "WIP":"work in progress",
            "POC":"proof of concept",
            "ping":"reach out"
        }
        suggestions=[]
        email_lower=email.lower()
        for jargon, replacement in replacements.items():
            if jargon.lower() in email_lower:
                suggestions.append(f"Consider replacing '{jargon}' with '{replacement}'")
                
        return "\n".join(suggestions) if suggestions else "No jargon or internal abbreviations detected." 
    
jt = ReplaceJargonsTool()
jt.run(original_email)
    
    

In [ ]:
from crewai import Agent, Task, Crew

email_assistant = Agent(
    role="Email Assistant Agent",
    goal="Improve emails and make them sound professional and clear",
    backstory="A highly experienced communication expert skilled in professional email writing",
    tools=[jt],
    verbose=True,
    llm=llm
)

email_task = Task(
    description=f"""Take the following rough email and rewrite it into a professional and polished version.
    Expand abbreviations:
    '''{original_email}''' """,
    agent=email_assistant,
    expected_output="A professional written email with proper formatting and content.",
)

crew = Crew(
    agents=[email_assistant],
    tasks=[email_task],
    verbose=True
)

result=crew.kickoff()
print(result)